# Aeon English Fluency — Zero-Cost Colab Campaign

**Halt state (source repo):** `FREE_COLAB_FLUENCY_BUNDLE_READY`.

This notebook trains Aeon toward English fluency on the free Colab GPU tier. It:

1. Mounts your Google Drive
2. Copies the source bundle from Drive into the temporary Colab filesystem
3. Installs pinned dependencies
4. Verifies every bundled file by SHA-256
5. Downloads WikiText-103 raw from the canonical S3 URL, verifies byte size + SHA-256, extracts
6. Detects CUDA (halts if unavailable)
7. Benchmarks Aeon for a short fixed token count and prints projected tokens/hour + estimated sessions
8. Trains **Stage 1** from the protected P2 checkpoint toward 100 million general-English tokens
9. Evaluates WikiText validation at fixed intervals
10. Trains **Stage 2** (Dolly-15k response-masked instruction tuning) starting from the best Stage-1 checkpoint
11. Evaluates on the locked **fresh_eval** subset (contamination-free)
12. Produces raw unedited generations after each major checkpoint

**Human review gate.** Dylan reviews the raw generations before any release approval. No automatic 'fluent' claim is emitted.

**Boundaries.** Never modifies parameters during inference. Never calls another model, teacher, corrector, retrieval system, LoRA, adapter, or fallback. Preserves architecture, clocks, K=16, margins, tokenizer, parameter count, state dimensions, state-dict topology.

**Dry-run.** Set `DRY_RUN = True` in cell 7 to verify the pipeline end-to-end (short benchmark + 5 training steps + one checkpoint + eval sample + generations) without a long training run.


## 1 · Mount Google Drive
This cell mounts Drive at `/content/drive` and fails loudly if the mount does not succeed. Every subsequent path is derived from the REAL mount point returned here — no `/content/drive/...` path is created on disk before this cell succeeds.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# Fail loudly if the mount did not produce a real MyDrive tree.
DRIVE_ROOT = '/content/drive'
assert os.path.isdir(DRIVE_ROOT), (
    f'Drive mount did not create {DRIVE_ROOT!r}. Re-run this cell.')
MYDRIVE = os.path.join(DRIVE_ROOT, 'MyDrive')
assert os.path.isdir(MYDRIVE), (
    f'Drive mount succeeded but MyDrive is missing at {MYDRIVE!r}. '
    'This usually means you cancelled the auth prompt or picked the '
    'wrong Google account. Re-run this cell.')
print('Drive mount OK ->', MYDRIVE)


## 2 · Copy the source bundle from Drive & install dependencies
Place `Aeon_English_Fluency_Colab_Bundle.zip` at the root of your Google Drive (MyDrive) first. This cell refuses to write anything to Drive until it has confirmed the mount is real.


In [ ]:
import os, shutil, subprocess, sys

# Guard: cell 1 must have run and succeeded first.
assert 'MYDRIVE' in globals(), (
    'MYDRIVE is not defined. Run cell 1 (Mount Google Drive) first.')
assert os.path.isdir(MYDRIVE), (
    f'Drive mount lost between cells; {MYDRIVE!r} no longer exists. '
    'Re-run cell 1.')

BUNDLE_ZIP = os.path.join(MYDRIVE, 'Aeon_English_Fluency_Colab_Bundle.zip')
WORK = '/content/aeon_bundle'
assert os.path.exists(BUNDLE_ZIP), (
    f'Bundle zip not found at {BUNDLE_ZIP}. Upload '
    'Aeon_English_Fluency_Colab_Bundle.zip to the root of your '
    'Google Drive (MyDrive) first.')
if os.path.exists(WORK): shutil.rmtree(WORK)
os.makedirs(WORK, exist_ok=True)
subprocess.check_call(['unzip', '-q', BUNDLE_ZIP, '-d', WORK])
print('bundle extracted at', WORK)

# Install pinned deps (torch already provided by Colab GPU runtime)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
    'sentencepiece==0.2.0', 'safetensors', 'pyyaml', 'numpy<2'])
print('deps installed')


## 3 · Verify every bundled file by SHA-256


In [ ]:
%cd /content/aeon_bundle
!python scripts/colab/verify_bundle.py --root .


## 4 · Download WikiText-103 raw (verified before extraction)
Canonical URL: `https://s3.amazonaws.com/research.metamind.io/wikitext/wikitext-103-raw-v1.zip`  
Expected byte size: `191,984,949`  
Expected SHA-256: `91c00ae287f0d699e18605c84afc9e45c192bc6b7797ff8837e5474655a33794`


In [ ]:
%cd /content/aeon_bundle
!python scripts/colab/download_wikitext103.py --out-dir /content/wikitext-103-raw


## 5 · Detect CUDA (halts if unavailable)


In [ ]:
%cd /content/aeon_bundle
!python scripts/colab/env_check.py


## 6 · Benchmark Aeon on this GPU


In [ ]:
%cd /content/aeon_bundle
!python scripts/colab/benchmark.py --root . --tokens 50000


## 7 · Training-run parameters
Set `DRY_RUN = True` to verify the pipeline without a long run.

**Google Drive checkpoint dir.** Adjust `DRIVE_RUN_DIR` if you want a different Drive location; checkpoints are what survives session termination.


In [ ]:
DRY_RUN = True

import os
assert 'MYDRIVE' in globals(), (
    'MYDRIVE is not defined. Run cell 1 (Mount Google Drive) first.')
assert os.path.isdir(MYDRIVE), (
    f'Drive mount lost; {MYDRIVE!r} no longer exists. Re-run cell 1.')

DRIVE_RUN_DIR = os.path.join(MYDRIVE, 'aeon_fluency_run')
STAGE1_CK_DIR = os.path.join(DRIVE_RUN_DIR, 'stage1_checkpoints')
STAGE2_CK_DIR = os.path.join(DRIVE_RUN_DIR, 'stage2_checkpoints')

STAGE1_TARGET_TOKENS  = 100_000_000
STAGE2_TARGET_TOKENS  =  10_000_000
CHECKPOINT_EVERY_TOK  =     250_000
CHECKPOINT_EVERY_SEC  =       1_800  # 30 minutes

# Set a wall-time cap per session so a single free-Colab session
# always halts cleanly with a fresh checkpoint on Drive.
SESSION_WALL_TIME_SEC = 42_000  # ~11.5 hours

if DRY_RUN:
    STAGE1_TARGET_TOKENS = 20_000
    STAGE2_TARGET_TOKENS = 5_000
    CHECKPOINT_EVERY_TOK = 5_000
    CHECKPOINT_EVERY_SEC = 120
    SESSION_WALL_TIME_SEC = 300

import os
os.makedirs(STAGE1_CK_DIR, exist_ok=True)
os.makedirs(STAGE2_CK_DIR, exist_ok=True)
print('DRY_RUN =', DRY_RUN)
print('STAGE1_TARGET_TOKENS =', STAGE1_TARGET_TOKENS)
print('STAGE2_TARGET_TOKENS =', STAGE2_TARGET_TOKENS)
print('STAGE1_CK_DIR =', STAGE1_CK_DIR)
print('STAGE2_CK_DIR =', STAGE2_CK_DIR)


## 8 · Stage 1 — train from protected P2 on WikiText-103 raw
Resumable. Checkpoint frequency: every `CHECKPOINT_EVERY_TOK` tokens OR every `CHECKPOINT_EVERY_SEC` seconds, whichever fires first. Session halts cleanly at `SESSION_WALL_TIME_SEC` with the current checkpoint written; **the next Colab session will resume automatically** from the latest checkpoint on Drive.

Re-run this cell as many times as needed until `STAGE1_TARGET_TOKENS` is reached.


In [ ]:
%cd /content/aeon_bundle
PARENT = 'runs/aeon_lbc1_P2/final.pt'   # protected P2 (never overwritten)
cmd = [
    'python', 'scripts/colab/train_stage.py',
    '--root', '.', '--stage', 'stage1',
    '--parent', PARENT,
    '--checkpoint-dir', STAGE1_CK_DIR,
    '--target-tokens', str(STAGE1_TARGET_TOKENS),
    '--checkpoint-every-tokens', str(CHECKPOINT_EVERY_TOK),
    '--checkpoint-every-seconds', str(CHECKPOINT_EVERY_SEC),
    '--wall-time-cap-seconds', str(SESSION_WALL_TIME_SEC),
]
if DRY_RUN: cmd.append('--dry-run')
import subprocess
subprocess.check_call(cmd)


## 9 · Stage 1 validation (WikiText valid — never test)


In [ ]:
%cd /content/aeon_bundle
import json, glob, os
cks = sorted(glob.glob(f'{STAGE1_CK_DIR}/checkpoint_*.pt'))
assert cks, 'No stage1 checkpoints yet — run cell 8 first.'
latest = cks[-1]
print('evaluating', latest)
import subprocess
subprocess.check_call(['python', 'scripts/colab/evaluate_and_generate.py',
    '--root', '.', '--mode', 'stage1_valid',
    '--checkpoint', latest,
    '--out', f'{STAGE1_CK_DIR}/eval_stage1_valid_latest.json'])
print(open(f'{STAGE1_CK_DIR}/eval_stage1_valid_latest.json').read())


## 10 · Stage 2 — Dolly-15k response-masked instruction tuning
Uses the best Stage-1 checkpoint as parent. Excludes every retired ID and the locked `fresh_eval` subset from the training pool.

Do not run this until Stage 1 has been trained to a reasonable validation loss (WikiText valid perplexity that Dylan judges acceptable).


In [ ]:
%cd /content/aeon_bundle
import glob
cks = sorted(glob.glob(f'{STAGE1_CK_DIR}/checkpoint_*.pt'))
assert cks, 'No stage1 checkpoints — cannot start stage2.'
best_stage1 = cks[-1]  # Or: whatever checkpoint Dylan chose after cell 9
print('stage2 parent =', best_stage1)
cmd = [
    'python', 'scripts/colab/train_stage.py',
    '--root', '.', '--stage', 'stage2',
    '--parent', best_stage1,
    '--checkpoint-dir', STAGE2_CK_DIR,
    '--target-tokens', str(STAGE2_TARGET_TOKENS),
    '--checkpoint-every-tokens', str(CHECKPOINT_EVERY_TOK),
    '--checkpoint-every-seconds', str(CHECKPOINT_EVERY_SEC),
    '--wall-time-cap-seconds', str(SESSION_WALL_TIME_SEC),
    '--seq-len', '256', '--batch-size', '4',
]
if DRY_RUN: cmd.append('--dry-run')
import subprocess
subprocess.check_call(cmd)


## 10b · Stage 2 checkpoint-selection signal (stage2_val — never a promotion gate)
Locked at manifest time (`stage2_val_lock_sha256` re-verified before scoring). Use this loss between Stage-2 checkpoints to pick a stopping point. It is NOT a promotion signal; that is `fresh_eval`'s single-use role.


In [ ]:
%cd /content/aeon_bundle
import glob
cks = sorted(glob.glob(f'{STAGE2_CK_DIR}/checkpoint_*.pt'))
assert cks, 'No stage2 checkpoints — run cell 10 first.'
latest = cks[-1]
import subprocess
subprocess.check_call(['python', 'scripts/colab/evaluate_and_generate.py',
    '--root', '.', '--mode', 'stage2_val',
    '--checkpoint', latest,
    '--out', f'{STAGE2_CK_DIR}/eval_stage2_val_latest.json'])
print(open(f'{STAGE2_CK_DIR}/eval_stage2_val_latest.json').read())


## 11 · Stage 2 evaluation — fresh_eval only (single-use promotion gate)
The evaluator verifies `fresh_eval_lock_sha256` before scoring; any drift aborts.

This value is one input to Dylan's approval decision — not a substitute for it. Do not use it to pick between checkpoints; that is `stage2_val`'s role above.


In [ ]:
%cd /content/aeon_bundle
import glob
cks = sorted(glob.glob(f'{STAGE2_CK_DIR}/checkpoint_*.pt'))
assert cks, 'No stage2 checkpoints — run cell 10 first.'
latest = cks[-1]
import subprocess
subprocess.check_call(['python', 'scripts/colab/evaluate_and_generate.py',
    '--root', '.', '--mode', 'stage2_fresh',
    '--checkpoint', latest,
    '--out', f'{STAGE2_CK_DIR}/eval_stage2_fresh_latest.json'])
print(open(f'{STAGE2_CK_DIR}/eval_stage2_fresh_latest.json').read())


## 12 · Raw generations after a major checkpoint
Deterministic greedy. Streamed decode is verified to equal one-shot decode. No rewriting. Dylan reviews these before any release approval.


In [ ]:
%cd /content/aeon_bundle
import glob, os
STAGE = 'stage2'  # switch to 'stage1' if you want Stage-1 samples
ck_dir = STAGE2_CK_DIR if STAGE == 'stage2' else STAGE1_CK_DIR
cks = sorted(glob.glob(f'{ck_dir}/checkpoint_*.pt'))
assert cks, f'No {STAGE} checkpoints.'
latest = cks[-1]
out_path = f'{ck_dir}/raw_generations_{STAGE}_latest.json'
import subprocess
subprocess.check_call(['python', 'scripts/colab/evaluate_and_generate.py',
    '--root', '.', '--mode', 'generate',
    '--checkpoint', latest,
    '--prompt-count', '25', '--max-new-tokens', '64',
    '--out', out_path])
print('wrote', out_path)


## Resume-later reminder
If your Colab session times out, just:

1. Reconnect to the same runtime (Runtime > Reconnect).
2. Re-run cells 1 – 7 (fast: mount, unzip, verify, GPU check).
3. Re-run cell 8 (Stage 1) — it resumes automatically from the latest checkpoint on Drive.
4. Later, run cells 10 – 12 for Stage 2 and evaluation.

The protected P2 checkpoint, tokenizer, architecture, K=16, margins, parameter count, and state-dict topology are preserved across every checkpoint and every session.
